# Cleaning 2.3 - Clean energy calculations

This notebook does the following:
    (1) Creates variables so that the dataset can be merged with the equipment calculators
    (2) Merges the dataset with the equipment calculators
    (3) Performs the energy calculations

## Set-up

In [ ]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from fill_missing_mode import fill_with_equipment_mode
from assign_set_temp import assign_set_temp

In [ ]:
# Load data
equipment = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_2.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [ ]:
# Load data dictionaries
equipment_data_dict = pd.read_excel(config.DATA_DICTIONARIES / "data_dictionary.xlsx", sheet_name="Equipment")

## (1) Prepare for merging with equipment calculators

In [ ]:
# Clean number variable

# If number is missing, set number to 1
equipment.loc[equipment["number"].isna(), "number"] = 1

# Make number integer
equipment["number"] = equipment["number"].astype(int)


In [ ]:
# Clean hours and days variables
print(equipment["equipment"].unique())

# If hours is missing or "Unknown", set hours to 9

#Add mask for equipment that has these variables, loop over values in tuple for each value in equipment type
has_day_hours_mask = equipment["equipment"].apply(lambda x: any(val in x for val in ("fc", "heater", "bath", "cryostat", "microbio", "glassware", "it")))
#always_on = ("fridge", "freezer", "ult") #Included for completeness

equipment.loc[(has_day_hours_mask & (equipment["hours"].isna()) | (equipment["hours"] == "Unknown")), "hours"] = 9

# If days is missing or "Unknown", set days to 255
equipment.loc[(has_day_hours_mask & (equipment["days"].isna()) | (equipment["days"] == "Unknown")), "days"] = 255

# Make hours and days numeric
equipment["hours"] = pd.to_numeric(equipment["hours"], errors="raise")
equipment["days"] = pd.to_numeric(equipment["days"], errors="raise")

In [ ]:
# Clean door openings
#Door opening is now in minutes!, and this impacts fridge, freezer, and ults
has_door_opening_mask = equipment["equipment"].apply(lambda x: any(val in x for val in ("freezer", "fridge", "ult")))
print(equipment.loc[has_door_opening_mask, "door_openings"].unique())

#Set to numeric and force Unknown values to NaN
equipment["door_openings"] = np.where(# For door opening values of the impacted equipment types that are unavailable or Unknown, use 0
    (equipment["door_openings"].isna() | equipment["door_openings"].isin(["Unknown"])) 
     & equipment["equipment"].isin(["freezer", "fridge", "ult"]), "0", equipment["door_openings"]
)
print(equipment.loc[has_door_opening_mask, "door_openings"].unique())
print(equipment["door_openings"].unique())
equipment["door_openings"] = pd.to_numeric(equipment["door_openings"], errors="coerce") #All invalid strings not pertaining to these variables not 

In [ ]:
# 1. Fix all data within its respective starting column

In [ ]:
# Clean FC variables

# Check unique values in hours_open
print("Unique values in hours_open before cleaning:")
print(equipment["hours_open"].unique())

# Fill missing/Unknown values for hours_open
equipment = fill_with_equipment_mode(
    equipment,
    value_col="hours_open",
    equipment_value="fc",
)

# Make hours_open numeric
equipment["hours_open"] = pd.to_numeric(equipment["hours_open"], errors="raise")

print(equipment["controller_type"].unique())
# Adjust "controller_type" variable which is "VAV" if "controller_type" is "Variable Air Volume (VAV)" or "Unknown" or missing, and "CAV" if "controller_type" is "Constant Air Volume (CAV)" ONLY if equipment is "fc"
equipment["controller_type"] = np.where(
    (equipment["controller_type"].isin(["Variable Air Volume (VAV)", "Unknown"]) | equipment["controller_type"].isna()) 
     & (equipment["equipment"] == "fc"),
    "VAV",
    np.where((equipment["controller_type"] == "Constant Air Volume (CAV)") & (equipment["equipment"] == "fc"),
             "CAV",
             equipment["controller_type"])
)
print(equipment["controller_type"].unique())

#Convert width to meters and numeric
print(equipment["sash_width"].unique())
#Convert relevant missing values to advised 
# equipment["sash_width"] = np.where(
#     (equipment["sash_width"].isna() | equipment["sash_width"].isin(["Unknown"]))
#     & (equipment["equipment"] == "fc"),
#     "1500", #TODO update with Andy advice (Unknown & fc + NAN to Andy advice)
#     equipment["sash_width"]
# )
equipment["sash_width"] = pd.to_numeric(equipment["sash_width"], errors="coerce")/1000
#check if in 0.5-3, as other values are not accepted by the website
equipment["sash_width"] = np.where((equipment["sash_width"] >= 0.5) & (equipment["sash_width"] <= 3), equipment["sash_width"], np.nan)

#This is a penalty variable
print(equipment["lifted"].unique())

#Input Andy advice for missing face velocity (0.15, 0.25) -> 0.2 as the missing value
print(equipment["face_velocity"].unique())
equipment["face_velocity"] == np.where(
    (equipment["face_velocity"].isna() | equipment["face_velocity"].isin(["Zentrum", "Irchel"])) & (equipment["equipment"] == "fc"),
    "0.2m/s",
    equipment["face_velocity"]
) #TODO? what to do about the units?


In [ ]:
# Clean Fridge Variables

##Match formatting and fix range of fridge size 250-425 -> 250-450
print(equipment["size_fridge"].unique())

#To extract just numbers, take piece before (~, drop )
#replace swaps out changed range value, added to a new column
equipment["size_fridge_2"] = equipment["size_fridge"].str.split("\\(~").str[1].str.split("\\)").str[0].replace({"250-425L": "250-450L"})
# Strings which contain fan are classified as such, convection likewise, and then remaining values should remain the same (nan/Unknown), added to a new column
equipment["size_fridge_1"] = np.where(
    equipment["size_fridge"].str.contains("fan", na=False), "Fan",
    np.where(equipment["size_fridge"].str.contains("convection", na=False), "Convection",
             equipment["size_fridge"])

)

print(equipment["size_fridge_1"].unique())
print(equipment["size_fridge_2"].unique())
#Note fridge has no door_opening option on website, but it still is in the code and was there previously

In [ ]:
# Clean Freezer variables

## Size and Type are now split out of size_freezer
print(equipment["size_freezer"].unique())
#80-130L -> 90-130L, 190-38L -> 190-380L
#To be added to Size, added to a new column
equipment["size_freezer_2"] = equipment["size_freezer"].str.split("\\(~").str[1].str.split("\\)").str[0].replace({"80-130L": "90-130L",
                                                                                                                  "190-38L": "190-380L"})
#To be added to type, added to a new column
equipment["size_freezer_1"] = np.where(
    equipment["size_freezer"].str.contains("Under bench", na=False), "Under Bench",
    np.where(equipment["size_freezer"].str.contains("Chest", na=False), "Chest",
             np.where(equipment["size_freezer"].str.contains("Tall", na=False), "Upright",
                      equipment["size_freezer"]))
)


#Drawer type has switched from Yes+Plastic/NoDrawers/No+Wire to Plastic/Wire
print(equipment["size_freezer_1"].unique())
print(equipment["size_freezer_2"].unique())

#Bound freezer temperatures
#Note: This is not read as a numeric variable
print(equipment["temp_freezer"].unique())
equipment["temp_freezer"] = equipment["temp_freezer"].replace({"-32": "-30",
                                                               "-18": "-20"})

#Icing is now a No or Yes question, so "a little bit" must become (no)
equipment["icing"] = np.where(
    equipment["icing"].str.contains("No|little"), "No, clear of ice",
    np.where(equipment["icing"].str.contains("Yes"), "Yes, iced up",
             equipment["icing"])
)

#Adjust freezer coolant
#Note: Ult refrigerant is stored in the ult_type variable, so no overlap
#Note
print(equipment["refrigerant"].unique())
equipment["refrigerant"] = np.where(
    equipment["refrigerant"].str.contains("HFC"), "HFC",
    np.where(equipment["refrigerant"].str.contains("HC"), "HC",
             equipment["refrigerant"])
)

In [ ]:
# Clean ULT variables

##Two type questions, one for engine and one for refridgerent + age. First type can limit the second type. Second type is classified as Age by Bell#ult_type splits into type, age takes the refrigerant
has_ult_type_mask = equipment["equipment"].apply(lambda x: any(val in x for val in ("ult")))
#Extract engine type, to be put in Type column, added to a new column
print(equipment["ult_type"].unique())
print("Engine response mode is " + equipment.loc[has_ult_type_mask, "ult_type"].mode().iloc[0]) #Most common answer is "Unknown"! can't use the function
print(equipment["ult_type"].value_counts())
#Most common known-type: "Cascade compressors with hydrocarbon refrigerants, under 10 years old"

equipment["ult_type_1"] = np.where(
    equipment["ult_type"].str.contains("Dual compressor", na=False), "Dual compressors",
    np.where(equipment["ult_type"].str.contains("Cascade", na=False), "Cascade compressors",
             np.where(equipment["ult_type"].str.contains("Stirling", na=False), "Stirling Engine",
                      np.where(equipment["ult_type"].str.contains("Unknown", na=False), "Cascade compressors", #See above
                               None)))
)
print("Engine type mode is " + equipment.loc[has_ult_type_mask, "ult_type_1"].mode().iloc[0])
#Extract age and refrigerant type to put into the age column, added to a new column
#TODO has redundancy. First deals with cases with clear matching, then deals with forced cases. Maybe swap, but those values are prevented
equipment["ult_type_2"] = np.where(
    (equipment["ult_type"].str.contains("hydrocarbon", na=False) & equipment["ult_type"].str.contains("under 10", na=False)), "<10yrs, HC",
    np.where((equipment["ult_type"].str.contains("hydrocarbon", na=False) & equipment["ult_type"].str.contains("over 10", na=False)), ">10yrs, HC",
             np.where((equipment["ult_type"].str.contains("hydrofluorocarbon", na=False) & equipment["ult_type"].str.contains("under 10", na=False)), "<10yrs, HFC",
                      np.where((equipment["ult_type"].str.contains("hydrofluorocarbon", na=False) & equipment["ult_type"].str.contains("over 10", na=False)), ">10yrs, HFC",
                               np.where((equipment["ult_type"].str.contains("Unknown", na=False) & equipment["ult_type"].str.contains("under 10", na=False)), "<10yrs, HC", #Matches website guideline
                                        np.where(equipment["ult_type"].str.contains("Stirling", na=False), "<10yrs, HC", #'Stirling engine' will always be "<10yrs, HC"
                                                 np.where((equipment["ult_type"].str.contains("Dual", na=False) & ~equipment["ult_type"].str.contains("years", na=False)), "Any Age, HC", #'Dual compressor with hydrocarbon refrigerants'
                                                          np.where((equipment["ult_type"].str.contains("Cascade", na=False) & ~equipment["ult_type"].str.contains("years", na=False)), "Any Age, HFC", #'Unknown with hydrofluorocarbon refrigerants', 'Cascade compressors with hydrofluorocarbon refrigerants', -> Any Age, HFC
                                                                   np.where(equipment["ult_type"].str.contains("Unknown", na=False), "<10yrs, HC",
                                                                   None))))))))
                                        # Cascade can not have HFC and age
                                        #Dual can not have HC and age
                                        #Stirling must be <10yrs, HC
)

#Temp
print(equipment["temp_ult"].unique())
#Run set_temp function to move all points to nearest valid option
equipment["temp_ult"] = equipment["temp_ult"].apply(
    lambda t: assign_set_temp(t, [-70, -75, -80])
)
print(equipment["temp_ult"].unique())

#Size of ULT must be reformatted
#Must fold Small single door (<500L) into smallest category
print(equipment["size_ult"].unique())
equipment["size_ult"] = np.where(
    equipment["size_ult"].str.contains("500"), "500-600L Tall", #will cover both categories
    np.where(equipment["size_ult"].str.contains("700"), "700-800L Tall",
             np.where(equipment["size_ult"].str.contains("900"), "800-900L Tall",
                      equipment["size_ult"]))
)
print(equipment["size_ult"].unique())

# Match formatting and collapse two non-damaged responses
# Policy on how to replace categories that are the same but overlap?
print(equipment["seals"].unique())
equipment["seals"] = np.where(
    equipment["seals"].isin(["Dirty or lightly iced", "Damaged, absent in places, or with a layer of ice"]), "No, seals are intact",
    np.where(equipment["seals"].isin(["Clean and in good condition"]), "Yes, seals are damaged",
             equipment["seals"])
)
print(equipment["seals"].unique())

# >= 150/Little spacing -> Good/Little
# print(equipment["spacing"].unique())
#Only two options, so matching here is exact
print(equipment["spacing"].unique())
equipment["spacing"] = equipment["spacing"].replace({"There's little space around the unit and we store items on top of the unit.": "Little / no spacing",
                                                     "≥150mm gap at the back and sides of the unit, with no items stored on top.": "Good spacing around unit (>= 150 mm on all sides)"})
print(equipment["spacing"].unique())

#Fix filter formatting
print(equipment["filter"].unique())
equipment["filter"] = np.where(
    equipment["filter"].isin(["It's clear"]), "No, filters are clear", # removed , "It's a little dirty" from folding in
    np.where(equipment["filter"].isin(["Most of it is clogged"]), "Yes, filters are clogged",
             equipment["filter"])#Leave 'No filter' option
)
print(equipment["filter"].unique())

print(equipment["ult_type_1"].unique())


In [ ]:
# Clean Glassware Variables

#Capacity reformating, just dropping first character
print(equipment["capacity_glassware"].unique())
equipment["capacity_glassware"] = equipment["capacity_glassware"].str.split("~").str[1]
print(equipment["capacity_glassware"].unique())

#tech is now age and has simpler formatting
print(equipment["tech"].unique())
equipment["tech"] = np.where(
    equipment["tech"].str.contains("Modern", na=False), "Newer", 
    np.where(equipment["tech"].str.contains("older", na=False), "Older", 
             equipment["tech"])
)

#Map to correct options that match wording
print(equipment["fan"].unique())
equipment["fan"] = np.where(
    equipment["fan"].str.contains("Yes"), "Yes, has a fan",
    np.where(equipment["fan"].str.contains("No"), "No fan",
             equipment["fan"]) #Fan has no I don't know option
)

# Remove I don't know option
equipment["temp_glassware"] = np.where(
    equipment["temp_glassware"].isin(["I don't know"]), np.nan, equipment["temp_glassware"]
)
#Assign values to nearest valid temperature
equipment["temp_glassware"] = equipment["temp_glassware"].apply(
    lambda t: assign_set_temp(t, [50, 60, 75])
)


In [ ]:
# Clean MSC Variables

#Already is a numeric value
print(equipment["width"].unique())
#Remove invalid option
equipment["width"] = equipment["width"].replace({1600: 1500})

#Age values must match format
print(equipment["age_microbio"].unique())
equipment["age_microbio"] = equipment["age_microbio"].replace({"Less than 5 years old": "<5 yrs",
                                                               "5-20 years old": "5-20yrs",
                                                               "More than 20 years old": ">20yrs"})
#Replace I don't know for age with mode
fill_with_equipment_mode(
    equipment,
    value_col="age_microbio",
    equipment_value="microbio",
    unknown_value="I don't know"
)

# Match No, recirculating / Yes, its ducted
print(equipment["ducting"].unique())
equipment["ducting"] = equipment["ducting"].replace({"No": "No, recirculating",
                                                     "Yes": "Yes, its ducted"})


In [ ]:
# Clean Incubator Variables

#No <150L, Add to 150-170L
print(equipment["capacity_incubator"].unique())
equipment["capacity_incubator"] = equipment["capacity_incubator"].replace({"<150L": "150-170L",
                                                                           "~150-170L": "150-170L",
                                                                           "~220-260L": "220-260L"})

#For age, ><20 -> Newer/Older
print(equipment["age_incubator"].unique())
equipment["age_incubator"] = equipment["age_incubator"].replace({"≤ 20 years": "Newer",
                                                                  "> 20 years": "Older"})

In [ ]:
# Clean water bath variables

# Fill missing/Unknown values for capacity_bath, heating, temp_bath, lid
for col in ["capacity_bath", "heating", "temp_bath", "lid"]:
    equipment = fill_with_equipment_mode(
        equipment,
        value_col=col,
        equipment_value="bath",
    )

# Modify "capacity_bath" to combine "10-12L" and "6-10L" into "10L". Currently mapped based on distance to median
print(equipment["capacity_bath"].unique())
equipment["capacity_bath"] = equipment["capacity_bath"].replace({"10-12L": "10L",
                                                                 "6-10L": "10L",
                                                                 "Less than 5L": "5-6L"})


# Set temperature for water bath rows, assigning to nearest valid value
#85 -> 90,/ 42, 40, 49, 46, 22(?) -> 37, /51 -> ?, 56, 58 -> 65 (Data -> SPARKHub val)
print(equipment["temp_bath"].unique())
equipment["temp_bath"] = np.where(
    equipment["temp_bath"].isin(["42", "40", "49", "46", "22"]), "37",
    np.where(equipment["temp_bath"].isin(["56", "58"]), "65",  #TODO? what to do about 51, which was mean(37, 65) 51 will be split in cleaning
             np.where(equipment["temp_bath"].isin(["85"]), "90",
                      equipment["temp_bath"]))
)
#Make sure string words are cleaned first, then should work (there are no strings)
equipment["temp_bath"] = equipment["temp_bath"].apply(
    lambda t: assign_set_temp(t, [37, 65, 90]) #Currently, 51 is set to na
)
print(equipment["temp_bath"].unique())

#Fix formatting of lid and heating
print(equipment["lid"].unique())
equipment["lid"] = equipment["lid"].replace({"Yes": "Lid is on during use",
                                             "No": "Lid is off during use"})

print(equipment["heating"].unique())
equipment["heating"] = equipment["heating"].replace({"Metal beads": "Beads"})

In [ ]:
# Clean cryostat variables

# Fill missing/Unknown values for temp_cryostat and sleep_mode
for col in ["temp_cryostat", "sleep_mode"]:
    equipment = fill_with_equipment_mode(
        equipment,
        value_col=col,
        equipment_value="cryostat",
    )

# Print unique values in sleep_mode
print(equipment["sleep_mode"].unique())

# Modify sleep_mode variable with "Energy Saving Mode Available"/"No Energy Saving Mode Available" based on sleep_mode
equipment["sleep_mode"] = np.where(
    (equipment["sleep_mode"].isin([
        "Yes, we use the unit 8am to 5pm, outside these hours it's in sleep mode",
        "Yes, but it's not used",
        "Yes, we use the unit 9am to 5pm, outside these hours it's in sleep mode"
        ]) | equipment["sleep_mode"].isna())    
        & (equipment["equipment"] == "cryostat"),
    "Energy Saving Mode Available", #All these values are treated as showing energy-saving mode is available now
    np.where((equipment["sleep_mode"] == "No, it does not")
             & (equipment["equipment"] == "cryostat"),
             "No Energy Saving Mode Available",
             equipment["sleep_mode"])
)

# Set temperatures only for cryostat rows
print(equipment["temp_cryostat"].unique())
# cryostat_mask = equipment["equipment"] == "cryostat"
equipment["temp_cryostat"] = equipment["temp_cryostat"].apply(
    lambda t: assign_set_temp(t, [-25, -20, -18]) #ensures only SPARKHub values are in data
)


In [ ]:
# Clean Heater Variables

#Match format, currently includes word "block"
print(equipment["blocks"].unique())
equipment["blocks"]= equipment["blocks"].str.extract(r'^(\d+)') #Extracts the numeric characters

# # 10(?), 45, 50 -> 37 / 60, 61, 64, 66, 76 -> 65 / 90 -> 95 / 120 -> 100 / 51, 80 -> ?
print(equipment["temp_heater"].unique())
equipment["temp_heater"] = np.where(
    equipment["temp_heater"].isin(["10", "45", "50"]), "37", #TODO 10 will be recleaned
    np.where(equipment["temp_heater"].isin(["60", "61", "64", "66", "76"]), "65",  #TODO what to do about 51, which was mean(37, 65) will be split in cleaning
             np.where(equipment["temp_heater"].isin(["90"]), "95", #TODO what to do about 80? will be split in cleaning
                      np.where(equipment["temp_heater"].isin(["120"]), "100", #TODO 120 will be rechecked
                               equipment["temp_heater"])))
)
#TODO Replace above with this once above tasks are checked
# equipment["temp_heater"] = np.where(
#     equipment["temp_heater"].isin(["Unknown"]), np.nan, equipment["temp_heater"]
# )
# equipment["temp_heater"] = equipment["temp_heater"].apply(
#     lambda t: assign_set_temp(t, [37, 65, 95, 100]) #ensures only SPARKHub values are in data
# )

print(equipment["temp_heater"].unique())

In [ ]:
# Clean it variables

# Check unique values of it_type
print(equipment["it_type"].unique())

# Fill missing/Unknown values for it_type
equipment = fill_with_equipment_mode(
    equipment,
    value_col="it_type",
    equipment_value="it",
)

## Monitor variable split for size (with the other equipments) and brightness (its own column)
print(equipment["monitor"].unique())
#Extract size related information and match to accepted values into a new column
equipment["monitor_size"] = np.where(
    equipment["monitor"].str.contains("Small", na=False), "19-22 inches",
    np.where(equipment["monitor"].str.contains("Medium", na=False), "24-27 inches",
             np.where(equipment["monitor"].str.contains("Large", na=False), "30+ inches",
                      np.where(equipment["monitor"].str.contains("does not", na=False), "No monitor", #to allow for the actual computers
                               equipment["monitor"]))) #Set to monitor value
)
#Extract brightness related information and match to accepted values into a new column
equipment["monitor_brightness"] = np.where(
    equipment["monitor"].str.contains("lowest", na=False), "Lowest",
    np.where(equipment["monitor"].str.contains("mid", na=False), "Mid",
             np.where(equipment["monitor"].str.contains("full", na=False), "Full",
                      equipment["monitor"])) #Set to monitor value
)

print(equipment["monitor_size"].unique())
print(equipment["monitor_brightness"].unique())

#Case matching is incorrect for it_type
print(equipment["it_type"].unique())
equipment["it_type"] = equipment["it_type"].replace({'High-powered desktop computer with graphics card': 'High-Powered Desktop Computer with Graphics Card'})

In [ ]:
# Map all variables to their new names
print(equipment["equipment"].unique())

# 3. Create columns to match and force variable types
## Process which shifts all the relevant data to the same column name that Isobel used.
## Changes regarding correct exact entries to be made afterwards
## Each section has a function which ensures only the relevant equipment have their values from these columns selected
## Other equipment gets None as the value

# Type = [controller_type, first halt size_fridge, first halt size_freezer, first half of ult_type, sleep_mode, it_type]
def equipment_type_func(row):
    if row["equipment"] == "fc":
        return row["controller_type"]
    elif row["equipment"] == "fridge":
        return row["size_fridge_1"]
    elif row["equipment"] == "freezer":
        return row["size_freezer_1"]
    elif row["equipment"] == "ult":
        return row["ult_type_1"]
    elif row["equipment"] == "cryostat":
        return row["sleep_mode"]
    elif row["equipment"] == "it":
        return row["it_type"]
    else:
        return None
equipment["Type"] = equipment.apply(lambda row: equipment_type_func(row), axis=1)


# Work_Surface_Width = [sash_width, width]
def equipment_width_func(row):
    if row["equipment"] == "fc":
        return row["sash_width"]
    elif row["equipment"] == "microbio":
        return row["width"]
    else:
        return None
equipment["Work_Surface_Width"] = equipment.apply(lambda row: equipment_width_func(row), axis=1)

# Size = [second half size_fridge, second half size_freezer, second half size_ult, capacity_bath, capacity_incubator, first half monitor]
def equipment_size_func(row):
    if row["equipment"] == "fridge":
        return row["size_fridge_2"]
    elif row["equipment"] == "freezer":
        return row["size_freezer_2"]
    elif row["equipment"] == "ult":
        return row["size_ult"]
    elif row["equipment"] == "bath":
        return row["capacity_bath"]
    elif row["equipment"] == "incubator":
        return row["capacity_incubator"]
    elif row["equipment"] == "it":
        return row["monitor_size"]
    else:
        return None
equipment["Size"] = equipment.apply(lambda row: equipment_size_func(row), axis=1)

# N_blocks = [blocks]
equipment["N_blocks"] = equipment["blocks"]
equipment["N_blocks"] = pd.to_numeric(equipment["N_blocks"], errors='coerce')

# Age = [secold half of ult_type (with age), tech, age_microbio, age_incubator]
def equipment_age_func(row):
    if row["equipment"] == "ult":
        return row["ult_type_2"]
    elif row["equipment"] == "cryostat":
        return row["tech"]
    elif row["equipment"] == "microbio":
        return row["age_microbio"]
    elif row["equipment"] == "incubator":
        return row["age_incubator"]
    else:
        return None
equipment["Age"] = equipment.apply(lambda row: equipment_age_func(row), axis=1)

# Set_Temp = [temp_freezer, temp_ult, temp_glassware, temp_cryostat, temp_bath, temp_heater]
def equipment_temp_func(row):
    if row["equipment"] == "freezer":
        return row["temp_freezer"]
    elif row["equipment"] == "ult":
        return row["temp_ult"]
    elif row["equipment"] == "glassware":
        return row["temp_glassware"]
    elif row["equipment"] == "cryostat":
        return row["temp_cryostat"]
    elif row["equipment"] == "bath":
        return row["temp_bath"]
    elif row["equipment"] == "heater":
        return row["temp_heater"]
    else:
        return None
equipment["Set_Temp"] = equipment.apply(lambda row: equipment_temp_func(row), axis=1)
equipment["Set_Temp"] = pd.to_numeric(equipment["Set_Temp"], errors='coerce')

# Brightness = [second half monitor]
equipment["Brightness"] = equipment["monitor_brightness"]

# Penalty indicators = [icing, drawers, seals, spacing, filter, fan, ducting, heating, lid, refrigerant]

# fc_specific = [face_velocity, lifted, surface, ] TODO look at code for this information

In [ ]:
# print(equipment.columns.values)
# print(equipment.shape)
print(equipment.iloc[:9, 57:64].dtypes)
#Examine each new column individually for data verification
def column_checker(colname):
    print("Counts for " + colname + "\n")
    print(equipment[colname].dtype.name)
    print(equipment[colname].unique())

column_checker("Type")
#We have no computer screen types? -> First merge on IT type, then on on the size/brightness etc
column_checker("Work_Surface_Width")
column_checker("Size")
column_checker("N_blocks")
column_checker("Age")
column_checker("Set_Temp")
column_checker("Brightness")
# print(equipment["Type"])


## Clean up and save processed data

In [ ]:
# Save processed data
# equipment.to_csv(config.PROCESSED_DATA / "panel_processed_3.csv", index =False)